In [ ]:
import pandas as pd

demo_J = pd.read_sas("DEMO_J.xpt", format="xport")
bpx_J = pd.read_sas("BPX_J.xpt", format="xport")
hdl_J = pd.read_sas("HDL_J.xpt", format="xport")
tchol_J = pd.read_sas("TCHOL_J.xpt", format="xport")
bmx_J = pd.read_sas("BMX_J.xpt", format="xport")
smq_J = pd.read_sas("SMQ_J.xpt", format="xport")
mcq_J = pd.read_sas("MCQ_J.xpt", format="xport")

In [ ]:
print(demo_J.shape)
print(hdl_J.shape)
print(tchol_J.shape)
print(bmx_J.shape)
print(smq_J.shape)
print(mcq_J.shape)


(9254, 46)
(7435, 3)
(7435, 3)
(8704, 21)
(6724, 37)
(8897, 76)


In [ ]:
def clean_nhanes_cycle(demo, bpx, hdl, tchol, bmx, smq, mcq, cycle_name):

    # --- Create CHD label ---
    mcq = mcq[mcq["MCQ160C"].isin([1, 2])]
    mcq["CHD"] = mcq["MCQ160C"].replace({1: 1, 2: 0})

    # --- Select necessary columns ---
    demo = demo[["SEQN", "RIDAGEYR", "RIAGENDR"]]
    bpx = bpx[["SEQN", "BPXSY1", "BPXDI1"]]
    hdl = hdl[["SEQN", "LBDHDD"]]
    tchol = tchol[["SEQN", "LBXTC"]]
    bmx = bmx[["SEQN", "BMXBMI"]]
    smq = smq[["SEQN", "SMQ020"]]
    mcq = mcq[["SEQN", "CHD"]]

    # --- Merge ---
    df = demo.merge(bpx, on="SEQN") \
             .merge(hdl, on="SEQN") \
             .merge(tchol, on="SEQN") \
             .merge(bmx, on="SEQN") \
             .merge(smq, on="SEQN") \
             .merge(mcq, on="SEQN")

    # --- Drop missing ---
    df = df.dropna()

    # --- Remove unrealistic values ---
    df = df[(df["BPXSY1"] > 50) & (df["BPXSY1"] < 250)]
    df = df[(df["BPXDI1"] > 30) & (df["BPXDI1"] < 150)]
    df = df[df["BMXBMI"] < 70]

    # --- Rename columns ---
    df = df.rename(columns={
        "RIDAGEYR": "age",
        "RIAGENDR": "sex",
        "BPXSY1": "sbp",
        "BPXDI1": "dbp",
        "LBXTC": "total_chol",
        "LBDHDD": "hdl",
        "BMXBMI": "bmi",
        "SMQ020": "smoking",
        "CHD": "chd"
    })

    # --- Encode categorical ---
    df["sex"] = df["sex"].replace({1: 1, 2: 0})
    df = df[df["smoking"].isin([1, 2])]
    df["smoking"] = df["smoking"].replace({1: 1, 2: 0})

    # --- Add cycle label ---
    df["cycle"] = cycle_name

    return df



In [ ]:
df_2017_2018 = clean_nhanes_cycle(
    demo_J, bpx_J, hdl_J, tchol_J, bmx_J, smq_J, mcq_J,
    "2017_2018"
)

df_2017_2018.shape

/tmp/ipython-input-3088076986.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  mcq["CHD"] = mcq["MCQ160C"].replace({1: 1, 2: 0})


(4281, 11)

In [ ]:
df_2017_2018["chd"].mean()
df_2017_2018.describe()



,SEQN,age,sex,sbp,dbp,hdl,total_chol,bmi,smoking,chd
count,4281.000000,4281.000000,4281.000000,4281.000000,4281.000000,4281.000000,4281.000000,4281.000000,4281.000000,4281.000000
mean,98313.780892,50.946041,0.491240,126.274702,72.698902,53.174258,188.552675,29.774118,0.426769,0.045550
std,2697.341419,17.602321,0.499982,19.273371,11.791381,15.616191,41.386271,7.178113,0.494666,0.208532
min,93709.000000,20.000000,0.000000,78.000000,32.000000,10.000000,76.000000,14.800000,0.000000,0.000000
25%,95992.000000,36.000000,0.000000,112.000000,66.000000,42.000000,160.000000,24.800000,0.000000,0.000000
50%,98326.000000,52.000000,0.000000,124.000000,72.000000,50.000000,185.000000,28.600000,0.000000,0.000000
75%,100692.000000,65.000000,1.000000,136.000000,80.000000,61.000000,214.000000,33.500000,1.000000,0.000000
max,102956.000000,80.000000,1.000000,224.000000,124.000000,178.000000,446.000000,67.700000,1.000000,1.000000


In [ ]:
df_2017_2018.to_csv("nhanes_2017_2018_clean.csv", index=False)

In [ ]:
import pandas as pd

df_2005_2006 = pd.read_csv("nhanes_2005_2006_clean.csv")
df_2007_2008 = pd.read_csv("nhanes_2007_2008_clean.csv")
df_2009_2010 = pd.read_csv("nhanes_2009_2010_clean.csv")
df_2011_2012 = pd.read_csv("nhanes_2011_2012_clean.csv")
df_2013_2014 = pd.read_csv("nhanes_2013_2014_clean.csv")
df_2015_2016 = pd.read_csv("nhanes_2015_2016_clean.csv")
df_2017_2018 = pd.read_csv("nhanes_2017_2018_clean.csv")

final_df = pd.concat([
    df_2005_2006,
    df_2007_2008,
    df_2009_2010,
    df_2011_2012,
    df_2013_2014,
    df_2015_2016,
    df_2017_2018
], axis=0)

final_df.shape

(32118, 11)

In [ ]:
final_df.to_csv("nhanes_2005_2018_unified.csv", index=False)

In [ ]:
df=pd.read_csv('nhanes_2005_2018_unified.csv')

In [ ]:
df.shape

(32118, 11)

In [ ]:
df.head()

,SEQN,age,sex,sbp,dbp,hdl,total_chol,bmi,smoking,chd,cycle
0,31131.0,44.0,0.0,144.0,74.0,39.0,105.0,30.90,0.0,0.0,2005_2006
1,31132.0,70.0,1.0,138.0,60.0,59.0,147.0,24.74,0.0,0.0,2005_2006
2,31134.0,73.0,1.0,130.0,68.0,49.0,186.0,30.63,0.0,0.0,2005_2006
3,31144.0,21.0,1.0,116.0,74.0,45.0,207.0,25.03,0.0,0.0,2005_2006
4,31150.0,79.0,1.0,144.0,74.0,81.0,181.0,28.93,1.0,0.0,2005_2006


In [ ]:
final_df["chd"].mean()
final_df["cycle"].value_counts()

,count
cycle,
2009_2010,5158
2007_2008,4855
2013_2014,4799
2015_2016,4766
2011_2012,4430
2017_2018,4281
2005_2006,3829
